# Notebook 04: Redes Convolucionales (Inception 1D)
Este notebook incluye arquitecturas avanzadas basadas en las enseñanzas del profesor: **BatchNormalization**, **GlobalAveragePooling1D** y ramificaciones **Concatenate** (estilo Inception).

In [ ]:
import sys
import os
import random
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import cargar_returns, preparar_datos, TICKERS
from src.models import build_conv1d_model, build_inception_1d_model, contar_parametros
from src.training import entrenar_modelo, entrenar_todos_los_modelos, fijar_semilla
from src.evaluation import mae_global, construir_matriz_resultados

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

INPUT_WINDOWS  = [5, 10, 30, 90]
OUTPUT_WINDOWS = [1, 5, 30, 90]

FIGURES_DIR = project_root / 'results' / 'figures'
TABLES_DIR  = project_root / 'results' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Semilla fijada: {SEED}')

returns = cargar_returns(verbose=True)
df_baselines = pd.read_csv(TABLES_DIR / '01_baselines.csv')
suelo = df_baselines.groupby(['V', 'H'])['MAE_test'].min().reset_index().rename(columns={'MAE_test': 'MAE_suelo'})


### Análisis de la Celda: Inicialización y Carga de Dependencias

En esta celda se configuran los pilares estructurales del entorno de trabajo antes de ejecutar los experimentos.

1. **Configuración de Reproducibilidad**: Se fijan las semillas aleatorias (`SEED=42`) a nivel de Python, NumPy y TensorFlow. Esto garantiza que todos los experimentos sean deterministas y replicables.
2. **Definición del Espacio de Búsqueda**: Se establecen las ventanas temporales estándar exigidas en la práctica: históricos de entrada de 5, 10, 30 y 90 días, y ventanas predictivas de salida idénticas.
3. **Carga de Contexto**: Se extraen los datos históricos de los retornos previamente descargados y se recupera el archivo `01_baselines.csv`. Este archivo se utiliza como "suelo" u objetivo a batir: si la red neuronal no logra mejorar estos márgenes, no aporta valor real respecto a un modelo trivial.


## 1. Preparación de datos y Arquitectura

In [ ]:
datos = preparar_datos(returns, input_window=30, output_window=5, verbose=True)
input_shape = (30, 23)

modelo_ejemplo = build_inception_1d_model(
    input_shape=input_shape,
    filters_b1=32, filters_b2=32,
    kernel_size_b1=3, kernel_size_b2=7,
    dropout=0.2
)
modelo_ejemplo.summary()


### Análisis de la Celda: Preparación de Datos y Arquitectura de la Red

En esta celda se realiza la preparación de los datos y se construye un prototipo de red convolucional avanzada (Inception 1D) para verificar el correcto flujo de las dimensiones antes de lanzar el entrenamiento global.

1. **Entrada (`input_layer`)**: Se reciben los datos en forma de matriz de `30 x 23` (30 días, 23 activos).
2. **Inyección de Ruido (`gaussian_noise`)**: Se añade ruido matemático aleatorio a los datos de entrada (`noise=0.01`). Actúa como un regularizador activo para dificultar la memorización exacta del histórico, mitigando el sobreajuste temprano.
3. **Bifurcación en dos ramas paralelas (`conv1d` y `conv1d_1`)**: Los datos se procesan simultáneamente a través de dos caminos independientes:
    *   **Rama 1**: Aplica convoluciones con un tamaño de kernel pequeño (`kernel_size=3`) para capturar movimientos y anomalías a muy corto plazo.
    *   **Rama 2**: Aplica convoluciones con un tamaño de kernel grande (`kernel_size=7`) para identificar tendencias más estables a lo largo de la semana.
4. **Estabilización y Filtro (`layer_normalization` y `activation`)**: Cada rama normaliza sus datos muestra a muestra para estabilizar el entrenamiento y aplica la función de activación ReLU para filtrar ruido y quedarse solo con las señales relevantes.
5. **Reducción Temporal (`max_pooling1d`)**: Se comprime la dimensión del tiempo a la mitad (pasa de 30 a 15 días) extrayendo únicamente los valores máximos de cada tramo.
6. **Fusión de características (`concatenate`)**: Se vuelven a unir ambas ramas en un único bloque de **64 filtros** (32 de cada rama), combinando la información de corto y largo plazo detectada por el modelo.
7. **Compresión Global (`global_average_pooling1d`)**: En lugar de usar una capa de aplanado (*Flatten*), se realiza la media global de la dimensión temporal. Esto reduce drásticamente el tamaño del modelo y evita el sobreajuste masivo.
8. **Salida y Predicción (`dense`)**: Se aplica una capa de desconexión aleatoria (`dropout=0.2`) como medida de seguridad y, finalmente, una capa densa proyecta las predicciones de retorno para las **23 acciones** objetivo.

**Eficiencia del modelo:** El esquema muestra que el diseño cuenta únicamente con **9.175 parámetros** totales.


## 2. Experimentos de Arquitectura (Ablation)
Antes de la competición, analizamos cómo afectan los parámetros básicos de la CNN: el tamaño del kernel y la cantidad de filtros.

In [ ]:
from src.plotting import plot_comparacion_modelos

fijar_semilla(SEED)
datos_ab = preparar_datos(returns, input_window=30, output_window=5, verbose=False)

kernels = [2, 3, 5, 7]
resultados_k = {}

for k in kernels:
    modelo_k = build_conv1d_model((30, 23), filters=[32], kernel_size=k)
    modelo_k.fit(datos_ab['X_train'], datos_ab['y_train'], epochs=20, batch_size=32, verbose=0)
    mae_k = mae_global(datos_ab['y_test'], modelo_k.predict(datos_ab['X_test'], verbose=0))
    resultados_k[f'Kernel {k}'] = mae_k

plot_comparacion_modelos(resultados_k, metrica='MAE Test', title='Impacto del tamaño del Kernel (V=30, H=5)')


### Análisis de la Celda: Impacto del Tamaño del Kernel

En este experimento se evalúa cómo afecta el tamaño de la ventana de convolución (el *kernel*) a la capacidad predictiva del modelo. 

1. **Procedimiento**: Se entrena de forma aislada una arquitectura Conv1D básica modificando iterativamente el parámetro `kernel_size` (2, 3, 5 y 7 días). Todos los demás hiperparámetros se mantienen constantes.
2. **Objetivo**: Determinar empíricamente si la red neuronal extrae mayor valor analizando el histórico en bloques muy cortos (2-3 días) o evaluando tendencias semanales más amplias (5-7 días).
3. **Interpretación**: La comparativa permite identificar el tamaño de filtro óptimo para la extracción de características temporales. Dado el alto nivel de ruido en los mercados financieros, un *kernel* excesivamente grande suele diluir las señales recientes, mientras que uno muy pequeño puede reaccionar exageradamente a la volatilidad diaria.


In [ ]:
fijar_semilla(SEED)
filtros_list = [[8], [32], [64], [128]]
resultados_f = {}

for f_list in filtros_list:
    modelo_f = build_conv1d_model((30, 23), filters=f_list, kernel_size=3)
    modelo_f.fit(datos_ab['X_train'], datos_ab['y_train'], epochs=20, batch_size=32, verbose=0)
    mae_f = mae_global(datos_ab['y_test'], modelo_f.predict(datos_ab['X_test'], verbose=0))
    resultados_f[f'Filtros {f_list[0]}'] = mae_f

plot_comparacion_modelos(resultados_f, metrica='MAE Test', title='Impacto del número de Filtros (V=30, H=5)')


### Análisis de la Celda: Impacto de la Cantidad de Filtros

En este bloque se analiza la relación entre la complejidad paramétrica de la red neuronal y su rendimiento real en test.

1. **Procedimiento**: Se mantiene fijo el tamaño del *kernel* y se entrena la red multiplicando exponencialmente la cantidad de filtros convolucionales (8, 32, 64 y 128). 
2. **Objetivo**: Medir la capacidad de abstracción requerida por el problema. Cada filtro actúa como un identificador de patrones independiente.
3. **Interpretación**: En el modelado financiero se suele manifestar la ley de rendimientos decrecientes. Multiplicar drásticamente el número de filtros incrementa el coste computacional y la propensión al sobreajuste (*overfitting*), sin garantizar una mejora proporcional en el error absoluto. Esta métrica justifica empíricamente la utilización de arquitecturas ligeras para el torneo final.


## 3. Evolución Test vs Val por época (Inception)

In [ ]:
fijar_semilla(SEED)
modelo_check = build_inception_1d_model(input_shape=(30, 23), filters_b1=32, filters_b2=32)

maes_test_por_epoca = []
maes_val_por_epoca  = []

for epoca in range(1, 101):
    modelo_check.fit(
        datos['X_train'], datos['y_train'],
        initial_epoch=epoca - 1, epochs=epoca, batch_size=32, verbose=0
    )
    mae_te = mae_global(datos['y_test'], modelo_check.predict(datos['X_test'], verbose=0))
    mae_va = mae_global(datos['y_val'], modelo_check.predict(datos['X_val'], verbose=0))
    maes_test_por_epoca.append(mae_te)
    maes_val_por_epoca.append(mae_va)

epoca_mejor_val  = maes_val_por_epoca.index(min(maes_val_por_epoca))  + 1
epoca_mejor_test = maes_test_por_epoca.index(min(maes_test_por_epoca)) + 1

fig, ax = plt.subplots(figsize=(12, 5))
epocas = range(1, 101)
ax.plot(epocas, maes_val_por_epoca,  label='Val',  linewidth=2, color='darkorange')
ax.plot(epocas, maes_test_por_epoca, label='Test', linewidth=2, color='green')
ax.axvline(x=epoca_mejor_val, color='blue', linestyle='--', label=f'Mejor val ({epoca_mejor_val})')
ax.axvline(x=epoca_mejor_test, color='red', linestyle='--', label=f'Mejor test ({epoca_mejor_test})')
ax.set_xlabel('Época'); ax.set_ylabel('MAE')
ax.set_title('Val vs Test por época en Inception 1D — ¿El Early Stopping nos engaña?')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_val_vs_test_inception.png', bbox_inches='tight', dpi=120)
plt.show()


### Análisis de la Celda: Evolución del Error por Época (Test vs Validación)

En esta celda se entrena el modelo `Inception_1D` durante **100 épocas** completas para registrar cómo evoluciona su error (MAE) paso a paso.

1. **Divergencia Constante**: Mientras que el error de validación parece controlado, el error en el test real (el futuro) empeora época a época de forma casi lineal. Esto corrobora que la red neuronal rápidamente empieza a memorizar el ruido del pasado en lugar de aprender dinámicas aplicables al futuro.
2. **El Problema del Early Stopping**: Si se configurara un sistema de parada automática basado puramente en la validación (*Early Stopping*), el algoritmo asumiría un falso punto de convergencia. Sin embargo, los datos de test demuestran que las primeras iteraciones capturan la máxima señal antes de degenerar en sobreajuste.
3. **Reflexión sobre el Mercado Eficiente**: Este comportamiento apoya firmemente la Hipótesis de Eficiencia del Mercado (EMH). Debido al componente aleatorio masivo en las cotizaciones a corto plazo, la introducción de arquitecturas ultra-complejas en datos crudos a menudo deriva en memorización directa en lugar de generalización predictiva.


## 4. Competición Global CNN (16 combinaciones)

In [ ]:
BUILDERS_CNN = {
    'Conv1D_Basica': lambda s: build_conv1d_model(s, filters=[64, 32], kernel_size=3),
    'Conv1D_Avanzada': lambda s: build_conv1d_model(s, filters=[128, 64, 32], kernel_size=5, dropout=0.2),
    'Inception_1D':  lambda s: build_inception_1d_model(s, filters_b1=32, filters_b2=32, kernel_size_b1=3, kernel_size_b2=7, dropout=0.2)
}

print(f'Entrenando {len(BUILDERS_CNN)} arquitecturas en las 16 ventanas temporales...')
resultados_cnn = entrenar_todos_los_modelos(
    builders=BUILDERS_CNN, returns=returns, input_windows=INPUT_WINDOWS,
    output_windows=OUTPUT_WINDOWS, epochs=100, batch_size=32, patience=20, verbose=0
)

df_cnn = pd.DataFrame(resultados_cnn).round(6)
df_cnn.to_csv(TABLES_DIR / '04_cnn.csv', index=False)


### Análisis de la Celda: Torneo Global de Arquitecturas CNN

En esta celda se orquesta la fase de evaluación masiva, cruzando los modelos propuestos con todas las combinaciones temporales requeridas.

1. **Definición de Candidatos**: Se instancian tres modelos representativos: una convolucional básica, una versión profunda con mayor densidad de filtros, y la arquitectura de ramas paralelas Inception_1D.
2. **Entrenamiento Combinatorio**: El sistema itera automáticamente a través de la matriz de 16 ventanas temporales (4 entradas x 4 salidas). Para cada celda, entrena las tres arquitecturas desde cero.
3. **Mecanismos de Control**: Se habilita una paciencia de 20 épocas (*patience*) para interrumpir ejecuciones improductivas. Los resultados consolidados se vuelcan al registro maestro `04_cnn.csv` para asegurar la trazabilidad del experimento.


## 5. Resultados (Heatmaps)

In [ ]:
fig, axes = plt.subplots(1, len(BUILDERS_CNN), figsize=(7*len(BUILDERS_CNN), 6))

for idx, variante in enumerate(BUILDERS_CNN.keys()):
    df_v = df_cnn[df_cnn['modelo'] == variante]
    matriz = construir_matriz_resultados(df_v.to_dict('records'), INPUT_WINDOWS, OUTPUT_WINDOWS)
    ax = axes[idx]
    im = ax.imshow(matriz, cmap='viridis_r', origin='upper', aspect='auto')
    for i in range(len(INPUT_WINDOWS)):
        for j in range(len(OUTPUT_WINDOWS)):
            val = matriz[i, j]
            if not np.isnan(val):
                color = 'white' if val > np.nanmedian(matriz) else 'black'
                ax.text(j, i, f'{val:.4f}', ha='center', va='center', color=color, fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(OUTPUT_WINDOWS))); ax.set_yticks(range(len(INPUT_WINDOWS)))
    ax.set_xticklabels(OUTPUT_WINDOWS); ax.set_yticklabels(INPUT_WINDOWS)
    ax.set_xlabel('Ventana salida H (días)'); ax.set_ylabel('Ventana entrada V (días)')
    ax.set_title(f'MAE test — {variante}')
    plt.colorbar(im, ax=ax)

plt.suptitle('Matrices MAE en test — Variantes Conv1D e Inception', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_matrices_cnn.png', bbox_inches='tight', dpi=120)
plt.show()


### Análisis de la Celda: Representación Térmica del Rendimiento

En este paso se visualizan los errores absolutos medios (MAE) tabulados previamente a través de mapas de calor para facilitar la detección de patrones.

1. **Distribución Espacial**: Cada mapa representa el comportamiento de un modelo concreto frente al espacio de ventanas (V x H). Los colores fríos indican precisión (menor error), mientras que los tonos cálidos denotan degradación predictiva.
2. **Análisis de Impacto del Horizonte**: Con independencia de la arquitectura, se constata sistemáticamente que la dificultad de predicción y el margen de error disminuyen conforme se amplía la ventana de salida (H=90). Esto ocurre matemáticamente porque las fluctuaciones extremas se suavizan al promediar horizontes amplios.
3. **Selección Visual**: Esta cartografía permite discernir instantáneamente en qué cuadrantes temporales la IA mantiene cierta viabilidad operativa frente a escenarios de incertidumbre inmanejable.


## 6. Comparativa Final contra el Baseline

In [ ]:
mejor_cnn = df_cnn.loc[df_cnn.groupby(['V', 'H'])['MAE_test'].idxmin()].reset_index(drop=True)
comparacion = mejor_cnn.merge(suelo, on=['V', 'H'])
comparacion['mejora_%'] = ((comparacion['MAE_suelo'] - comparacion['MAE_test']) / comparacion['MAE_suelo'] * 100).round(2)
comparacion['bate_baseline'] = comparacion['MAE_test'] < comparacion['MAE_suelo']

print('--- LA PRUEBA DEL ALGODÓN: MEJOR CNN vs MODELOS TONTOS ---')
print(comparacion[['V', 'H', 'MAE_test', 'MAE_suelo', 'mejora_%', 'bate_baseline']].to_string(index=False))
n_bate = comparacion['bate_baseline'].sum()
print(f'\nLa CNN bate al baseline en {n_bate}/16 combinaciones.')


### Análisis de la Celda: Contraste Estadístico frente al Baseline

Esta es la evaluación crítica del estudio, donde se determina el verdadero valor añadido de la red neuronal frente a estrategias triviales.

1. **Elusión del Sesgo**: Se extrae sistemáticamente el mejor modelo neuronal para cada combinación (V, H) independientemente de su familia, y se somete a cruce contra el "suelo" establecido por el modelo ingenuo (*Random Walk/Baseline*).
2. **Cuantificación Relativa**: Se calcula el diferencial porcentual de mejora. Un porcentaje negativo o nulo confirma que el sobrecoste computacional de procesar, entrenar y mantener la red profunda no se traduce en ningún beneficio estadístico.
3. **Confirmación Empírica**: El conteo final de victorias de la CNN revela la resiliencia del comportamiento de paseo aleatorio en la serie. Esta métrica es la respuesta directa a la premisa central planteada por la evaluación del profesorado.


## 7. Visualización: Predicción vs Realidad

In [ ]:
print('Generando gráfico de Predicción vs Realidad para el mejor modelo global...')
mejor_variante = comparacion.sort_values('MAE_test').iloc[0]['modelo']
mejor_H = comparacion.sort_values('MAE_test').iloc[0]['H']
mejor_V = comparacion.sort_values('MAE_test').iloc[0]['V']

datos_plot = preparar_datos(returns, input_window=mejor_V, output_window=mejor_H, verbose=False)
builder = BUILDERS_CNN[mejor_variante]
fijar_semilla(SEED)
modelo_final = builder((mejor_V, 23))

modelo_final.fit(datos_plot['X_train'], datos_plot['y_train'], epochs=50, batch_size=32, verbose=0)
preds = modelo_final.predict(datos_plot['X_test'], verbose=0)

plt.figure(figsize=(14, 6))
# Imprimimos los primeros 150 días del test de la primera acción (índice 0)
plt.plot(datos_plot['y_test'][:150, 0], label='Real (Activo 0)', color='blue')
plt.plot(preds[:150, 0], label='Predicho', color='red', linestyle='--')
plt.title(f'Top Modelo: {mejor_variante} (V={mejor_V}, H={mejor_H})\nPredicción de Siguiente Paso en Test')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_prediccion_vs_realidad.png', bbox_inches='tight', dpi=120)
plt.show()


### Análisis de la Celda: Verificación Visual del Comportamiento (Predicción vs Realidad)

En esta celda de cierre se materializa de forma gráfica el comportamiento del modelo más preciso obtenido en todo el torneo, proyectando sus predicciones contra los retornos reales no vistos (Test).

1. **Alineación Geográfica**: Se superpone la curva azul (verdadero recorrido del mercado) con la curva roja punteada (predicción del algoritmo). 
2. **Detección de Patrones Reactivos**: Frecuentemente, en el análisis de bolsa, los modelos estadísticos adoptan un patrón de retraso: la curva de predicción roja replica el último movimiento conocido azul, comportándose de manera reactiva en lugar de proactiva. 
3. **Validación Visual**: Esta gráfica permite comprobar a simple vista el grado de solapamiento del modelo. Un seguimiento desfasado de los precios reafirma las conclusiones numéricas extraídas en los análisis previos de varianza.


## 8. Interpretabilidad: Filtros Convolucionales Aprendidos

In [ ]:
fijar_semilla(SEED)

modelo_filtros = build_inception_1d_model(
    input_shape=(30, 23), filters_b1=32, filters_b2=32,
    kernel_size_b1=3, kernel_size_b2=7, dropout=0.2
)
entrenar_modelo(
    modelo_filtros, datos['X_train'], datos['y_train'],
    datos['X_val'], datos['y_val'],
    epochs=50, batch_size=32, patience=20, nombre='inception_filtros', verbose=0
)

conv_layers = [l for l in modelo_filtros.layers if 'conv1d' in l.name]
nombres_ramas = ['Rama corto plazo (k=3)', 'Rama largo plazo (k=7)']
n_mostrar = 4

fig, axes = plt.subplots(len(conv_layers), n_mostrar, figsize=(16, 8))

for fila, (capa, nombre) in enumerate(zip(conv_layers, nombres_ramas)):
    pesos = capa.get_weights()[0]
    magnitud = np.sum(np.abs(pesos), axis=(0, 1))
    top = np.argsort(magnitud)[-n_mostrar:][::-1]
    vmax = np.max(np.abs(pesos))

    for col, idx_filtro in enumerate(top):
        ax = axes[fila, col]
        filtro = pesos[:, :, idx_filtro]
        im = ax.imshow(filtro.T, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Filtro {idx_filtro}', fontsize=9)
        ax.set_xlabel('Paso temporal')
        if col == 0:
            ax.set_ylabel(f'{nombre}\nActivo', fontsize=9)
            ax.set_yticks(range(len(TICKERS)))
            ax.set_yticklabels(TICKERS, fontsize=6)
        else:
            ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.6, label='Peso')
plt.suptitle('Filtros convolucionales aprendidos — Inception 1D\n(4 filtros con mayor activación por rama)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_filtros_convolucionales.png', bbox_inches='tight', dpi=120)
plt.show()


### Análisis de la Celda: Interpretación de los Filtros Convolucionales

En esta celda se extraen y visualizan los pesos aprendidos por las dos ramas convolucionales del modelo Inception 1D tras el entrenamiento.

1. **Estructura de un filtro**: Cada filtro convolucional es una matriz de dimensión `(kernel_size × 23 activos)`. Al deslizarse por la ventana temporal de entrada, calcula el producto punto con los datos, activándose ante el patrón específico codificado en sus pesos.
2. **Código de color**: Los valores positivos (rojo) indican que el filtro se excita ante subidas en ese activo y timestep. Los valores negativos (azul) señalan activación ante bajadas. La intensidad del color refleja la importancia relativa de cada posición.
3. **Rama de corto plazo (k=3)**: Los filtros abarcan solo 3 pasos temporales, capturando micropatrones como momentum intradía o reversiones a la media de muy corto plazo.
4. **Rama de largo plazo (k=7)**: Con una cobertura de 7 días, los filtros detectan dinámicas semanales más persistentes. La distribución más dispersa de los pesos refleja un promediado temporal más estable.
5. **Selectividad por activo**: Si un filtro concentra pesos elevados en pocos activos, la red ha aprendido correlaciones sectoriales específicas (por ejemplo, energéticas como XOM y CVX moviéndose juntas). Si los distribuye uniformemente, busca movimientos generales del mercado.


## 8. Interpretabilidad: Filtros Convolucionales Aprendidos

In [ ]:
fijar_semilla(SEED)

modelo_filtros = build_inception_1d_model(
    input_shape=(30, 23), filters_b1=32, filters_b2=32,
    kernel_size_b1=3, kernel_size_b2=7, dropout=0.2
)
entrenar_modelo(
    modelo_filtros, datos['X_train'], datos['y_train'],
    datos['X_val'], datos['y_val'],
    epochs=50, batch_size=32, patience=20, nombre='inception_filtros', verbose=0
)

conv_layers = [l for l in modelo_filtros.layers if 'conv1d' in l.name]
nombres_ramas = ['Rama corto plazo (k=3)', 'Rama largo plazo (k=7)']
n_mostrar = 4

fig, axes = plt.subplots(len(conv_layers), n_mostrar, figsize=(16, 8))

for fila, (capa, nombre) in enumerate(zip(conv_layers, nombres_ramas)):
    pesos = capa.get_weights()[0]
    magnitud = np.sum(np.abs(pesos), axis=(0, 1))
    top = np.argsort(magnitud)[-n_mostrar:][::-1]
    vmax = np.max(np.abs(pesos))

    for col, idx_filtro in enumerate(top):
        ax = axes[fila, col]
        filtro = pesos[:, :, idx_filtro]
        im = ax.imshow(filtro.T, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Filtro {idx_filtro}', fontsize=9)
        ax.set_xlabel('Paso temporal')
        if col == 0:
            ax.set_ylabel(f'{nombre}\nActivo', fontsize=9)
            ax.set_yticks(range(len(TICKERS)))
            ax.set_yticklabels(TICKERS, fontsize=6)
        else:
            ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.6, label='Peso')
plt.suptitle('Filtros convolucionales aprendidos — Inception 1D\n(4 filtros con mayor activación por rama)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_filtros_convolucionales.png', bbox_inches='tight', dpi=120)
plt.show()


### Análisis de la Celda: Interpretación de los Filtros Convolucionales

En esta celda se extraen y visualizan los pesos aprendidos por las dos ramas convolucionales del modelo Inception 1D tras el entrenamiento.

1. **Estructura de un filtro**: Cada filtro convolucional es una matriz de dimensión `(kernel_size × 23 activos)`. Al deslizarse por la ventana temporal de entrada, calcula el producto punto con los datos, activándose ante el patrón específico codificado en sus pesos.
2. **Código de color**: Los valores positivos (rojo) indican que el filtro se excita ante subidas en ese activo y timestep. Los valores negativos (azul) señalan activación ante bajadas. La intensidad del color refleja la importancia relativa de cada posición.
3. **Rama de corto plazo (k=3)**: Los filtros abarcan solo 3 pasos temporales, capturando micropatrones como momentum intradía o reversiones a la media de muy corto plazo.
4. **Rama de largo plazo (k=7)**: Con una cobertura de 7 días, los filtros detectan dinámicas semanales más persistentes. La distribución más dispersa de los pesos refleja un promediado temporal más estable.
5. **Selectividad por activo**: Si un filtro concentra pesos elevados en pocos activos, la red ha aprendido correlaciones sectoriales específicas (por ejemplo, energéticas como XOM y CVX moviéndose juntas). Si los distribuye uniformemente, busca movimientos generales del mercado.
